# Extra check 1 - Direct `<unk>`: text-to-text diagnostic

Does the `<unk>` in the Direct system need the speech (accent, audio), or does it also appear when the model gets only the English text? We give the same SeamlessM4T v2-large model the English text instead of the audio (same pinned revision as the main Direct run, greedy decoding, `max_new_tokens=256`, target `cmn`). If `<unk>` still appears for most of the same sentences, the audio is not necessary for it and the text translation/generation process plausibly plays a role; if it mostly disappears, the speech path matters. Either way this does not identify the exact mechanism. We use two text inputs: the reference English sentence and Whisper's transcript, in batches of 32 (the original run did one clip at a time, so a few outputs may differ). It is a post-hoc diagnostic; the original 4,200 Direct predictions are not changed.

You need a GPU runtime (a Colab T4 is fine), a Hugging Face login, and `merged_3system_with_metrics.csv` uploaded to the Colab working folder.

In [ ]:
%pip install -q "transformers>=4.40,<5" sentencepiece protobuf sacrebleu pandas tqdm

In [ ]:
import json, re, time
from pathlib import Path
import numpy as np, pandas as pd, torch, transformers, sacrebleu
from tqdm.auto import tqdm
from transformers import AutoProcessor, SeamlessM4Tv2ForTextToText

MODEL_ID = "facebook/seamless-m4t-v2-large"
MODEL_REVISION = "5f8cc790b19fc3f67a61c105133b20b34e3dcb76"   # same as the main Direct run
TARGET_LANGUAGE = "cmn"
SOURCE_LANGUAGE = "eng"
GENERATION_KWARGS = {"num_beams": 1, "do_sample": False, "max_new_tokens": 256}   # same as main run

MERGED_CSV = Path("merged_3system_with_metrics.csv")   # <- change if you put it elsewhere
OUT_DIR = Path("runs") / f"direct_text_to_text_diagnostic_{int(time.time())}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# BATCH_SIZE = 1 is closest to the original one-clip-at-a-time run (slower).
# Larger batches are faster but padding can change a few outputs slightly.
BATCH_SIZE = 32
SMOKE_TEST = False        # True = only 64 sentences, to check that everything runs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print("device:", DEVICE, "| dtype:", DTYPE, "| transformers:", transformers.__version__)
if DEVICE.type != "cuda":
    print("WARNING: no GPU found - this will be slow and not identical to the T4/FP16 run.")

In [ ]:
from huggingface_hub import login
login()   # paste your own Hugging Face token when asked; do not commit it

In [ ]:
df = pd.read_csv(MERGED_CSV)
need = ["id", "sentence", "primary_accent", "asr_transcript", "direct_translation", "reference_translation_zh", "direct_has_unk"]
assert all(c in df.columns for c in need), "unexpected columns"
assert len(df) == 4200 and df.id.is_unique
df["asr_transcript"] = df.asr_transcript.fillna("").astype(str)
print(df.shape, "| unique reference sentences:", df.sentence.nunique(), "| unique ASR texts:", df.asr_transcript.nunique())

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = SeamlessM4Tv2ForTextToText.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE)
model.to(DEVICE).eval()
UNK_ID = processor.tokenizer.unk_token_id
print("unk_token_id:", UNK_ID)

In [ ]:
UNK_RE = re.compile(r"<\s*unk\s*>", re.IGNORECASE)   # also catches '<unk >' variants

def translate_texts(texts, batch_size=BATCH_SIZE):
    """Translate a list of unique English strings; returns {text: (translation, n_special_unk_ids, n_tokens)}."""
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # similar lengths per batch
    out = {}
    for s in tqdm(range(0, len(order), batch_size)):
        idx = order[s:s + batch_size]
        batch = [texts[i] for i in idx]
        enc = processor(text=batch, src_lang=SOURCE_LANGUAGE, return_tensors="pt", padding=True).to(DEVICE)
        with torch.inference_mode():
            gen = model.generate(**enc, tgt_lang=TARGET_LANGUAGE, **GENERATION_KWARGS)
        seqs = gen if torch.is_tensor(gen) else gen.sequences
        decoded = processor.batch_decode(seqs, skip_special_tokens=True)
        for i, ids, t in zip(idx, seqs.cpu().tolist(), decoded):
            out[texts[i]] = (t.strip(), sum(x == UNK_ID for x in ids), len(ids))
    return out

ref_texts = sorted(df.sentence.unique())
asr_texts = sorted(t for t in df.asr_transcript.unique() if t.strip())
if SMOKE_TEST:
    ref_texts, asr_texts = ref_texts[:64], asr_texts[:64]
t0 = time.time()
ref_out = translate_texts(ref_texts)
asr_out = translate_texts(asr_texts)
print(f"done in {(time.time()-t0)/60:.1f} min")

In [ ]:
def get(out, text, k):
    return out.get(text, (None, None, None))[k]

res = df[need].copy()
res["text_ref_translation"] = res.sentence.map(lambda t: get(ref_out, t, 0))
res["text_ref_special_unk_ids"] = res.sentence.map(lambda t: get(ref_out, t, 1))
res["text_asr_translation"] = res.asr_transcript.map(lambda t: get(asr_out, t, 0))
res["text_asr_special_unk_ids"] = res.asr_transcript.map(lambda t: get(asr_out, t, 1))
if SMOKE_TEST:
    res = res.dropna(subset=["text_ref_translation"])

res["speech_unk"]    = res.direct_translation.fillna("").map(lambda s: bool(UNK_RE.search(s)))
res["text_ref_unk"]  = res.text_ref_translation.fillna("").map(lambda s: bool(UNK_RE.search(s)))
res["text_asr_unk"]  = res.text_asr_translation.fillna("").map(lambda s: bool(UNK_RE.search(s)))
res.to_csv(OUT_DIR / "text_to_text_predictions.csv", index=False)
print("saved", OUT_DIR / "text_to_text_predictions.csv", "| rows:", len(res))

In [ ]:
# 1) How often does <unk> appear for each input type?
n = len(res)
rates = pd.DataFrame({
    "input": ["Speech (original Direct run, regex)", "English text (reference sentence)", "English text (Whisper transcript)"],
    "clips_with_unk": [res.speech_unk.sum(), res.text_ref_unk.sum(), res.text_asr_unk.sum()],
})
rates["rate_pct"] = (100 * rates.clips_with_unk / n).round(1)
print(rates.to_string(index=False))
print("\nspecial UNK token id generated in text runs:", int(res.text_ref_special_unk_ids.sum()), int(res.text_asr_special_unk_ids.sum()))
print("original exact-string count (full run should be 508):", int(res.direct_has_unk.sum()))

# 2) Do the same sentences fail?
ct = pd.crosstab(res.speech_unk, res.text_ref_unk, rownames=["speech <unk>"], colnames=["text <unk>"])
print("\n", ct)
a, b = res.speech_unk, res.text_ref_unk
p1 = b[a].mean(); p0 = b[~a].mean()
print(f"\nP(text <unk> | speech <unk>)    = {p1:.3f}")
print(f"P(text <unk> | speech no <unk>) = {p0:.3f}")
print(f"Jaccard overlap = {(a & b).sum() / (a | b).sum():.3f}")

In [ ]:
# 3) By accent
by = res.groupby("primary_accent").agg(clips=("id", "size"), speech_unk=("speech_unk", "mean"),
                                       text_ref_unk=("text_ref_unk", "mean"), text_asr_unk=("text_asr_unk", "mean"))
(by[["speech_unk", "text_ref_unk", "text_asr_unk"]] * 100).round(1).assign(clips=by.clips)

In [ ]:
# 4) Translation quality (sentence chrF++, same settings as the team's CSV)
chrf = sacrebleu.metrics.CHRF(word_order=2)
sc = lambda h, r: chrf.sentence_score(h if isinstance(h, str) else "", [r]).score
res["chrfpp_speech"] = [sc(h, r) for h, r in zip(res.direct_translation, res.reference_translation_zh)]
res["chrfpp_text_ref"] = [sc(h, r) for h, r in zip(res.text_ref_translation, res.reference_translation_zh)]
print(res.groupby("speech_unk")[["chrfpp_speech", "chrfpp_text_ref"]].mean().round(2))
print("\nall clips:", res[["chrfpp_speech", "chrfpp_text_ref"]].mean().round(2).to_dict())
res.to_csv(OUT_DIR / "text_to_text_predictions.csv", index=False)

In [ ]:
# 5) Examples
ex = res[res.speech_unk & res.text_ref_unk].drop_duplicates("sentence").head(10)
print("---- <unk> with speech AND with text ----")
for r in ex.itertuples():
    print("EN    :", r.sentence); print("SPEECH:", r.direct_translation); print("TEXT  :", r.text_ref_translation); print()
ex2 = res[res.speech_unk & ~res.text_ref_unk].drop_duplicates("sentence").head(10)
print("---- <unk> with speech but NOT with text ----")
for r in ex2.itertuples():
    print("EN    :", r.sentence); print("SPEECH:", r.direct_translation); print("TEXT  :", r.text_ref_translation); print()

In [ ]:
summary = {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "device": str(DEVICE), "dtype": str(DTYPE),
    "generation": GENERATION_KWARGS, "batch_size": BATCH_SIZE, "smoke_test": SMOKE_TEST,
    "n_clips": int(len(res)),
    "speech_unk_rate_pct": float(100 * res.speech_unk.mean()),
    "text_ref_unk_rate_pct": float(100 * res.text_ref_unk.mean()),
    "text_asr_unk_rate_pct": float(100 * res.text_asr_unk.mean()),
    "p_text_unk_given_speech_unk": float(p1), "p_text_unk_given_speech_no_unk": float(p0),
    "note": "Post-hoc diagnostic. Original 4,200 Direct predictions unchanged.",
}
(OUT_DIR / "run_summary_unk_extra_1_text.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))